# Chapter 11 — Support Vector Machines I: Discriminant Functions, Decision Boundaries & the Perceptron

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part2_classic_ml/ch11_support_vector_machines_i/ch11_notebook.ipynb)

Companion notebook for **Chapter 11** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Every classifier carves the input space into class regions. The border between them is the decision boundary.
- Non-discriminative methods (K-NN; generative Bayes models) model each class. Discriminative methods (perceptron, logistic regression, SVM) learn the boundary.
- In a Gaussian Bayes classifier, shared covariances give linear boundaries and different covariances give curved ones.
- A linear boundary is g(x) = wᵀx + w₀ = 0, and the sign of g(x) gives the class.
- The perceptron nudges its weights on every mistake and converges if the data is linearly separable.
- The perceptron fails on non-separable data, and the line it finds is not the best one. SVMs fix both problems.

## Example 1: Discriminant Functions and Bayes Decision Boundaries

The first part evaluates g(x) = x₁ + x₂ − 1 at a few example points and reports which side each one is on. The second part checks the three-case theory. When two classes share a covariance matrix, a linear boundary (LDA) does just as well as a quadratic one (QDA), and the quadratic term W is almost zero. When the covariances differ, one class is tight and the other spread out, so the quadratic boundary wins (93.0% vs 87.5%) and W is clearly non-zero, which means the boundary curves.

In [1]:
# Chapter 11 - Example 1: Discriminant functions - which side of the line is a point on?
# And how the covariance decides whether a Bayes boundary is straight or curved.
import numpy as np
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis,
                                           QuadraticDiscriminantAnalysis)

# 1. The line x1 + x2 - 1 = 0 has discriminant g(x) = x1 + x2 - 1
g = lambda x: x[0] + x[1] - 1
for p in [(1, 0), (0, 1), (0, 0), (1, 1), (0.2, 0.3)]:
    v = g(p)
    side = "on the line" if v == 0 else "positive side" if v > 0 else "negative side"
    print(f"g{p} = {v:+.1f}  -> {side}")

# 2. Bayes classifier with Gaussian classes
rng = np.random.default_rng(0)
def sample(mean, cov, n=300): return rng.multivariate_normal(mean, cov, n)

cases = {
    "Same covariance (Case 2)": ([[2, 1], [1, 2]], [[2, 1], [1, 2]]),
    "Different covariance (Case 3)": ([[0.3, 0], [0, 0.3]], [[4, 0], [0, 4]]),
}
for name, (c0, c1) in cases.items():
    X = np.vstack([sample([0, 0], c0), sample([2, 2], c1)])
    y = np.r_[np.zeros(300), np.ones(300)]
    lda = LinearDiscriminantAnalysis().fit(X, y)       # assumes a shared covariance -> line
    qda = QuadraticDiscriminantAnalysis().fit(X, y)    # one covariance per class -> curve
    print(f"\n{name}")
    print(f"  Linear boundary accuracy   : {lda.score(X, y):.3f}")
    print(f"  Quadratic boundary accuracy: {qda.score(X, y):.3f}")
    # Size of the x^T W x term: W = inverse(cov1) - inverse(cov0)
    W = np.linalg.inv(np.cov(X[y == 1].T)) - np.linalg.inv(np.cov(X[y == 0].T))
    print(f"  Size of quadratic term |W| = {np.abs(W).max():.2f}"
          f"  -> boundary is {'almost straight' if np.abs(W).max() < 0.3 else 'curved'}")

g(1, 0) = +0.0  -> on the line
g(0, 1) = +0.0  -> on the line
g(0, 0) = -1.0  -> negative side
g(1, 1) = +1.0  -> positive side
g(0.2, 0.3) = -0.5  -> negative side

Same covariance (Case 2)
  Linear boundary accuracy   : 0.802
  Quadratic boundary accuracy: 0.805
  Size of quadratic term |W| = 0.05  -> boundary is almost straight

Different covariance (Case 3)
  Linear boundary accuracy   : 0.875
  Quadratic boundary accuracy: 0.930
  Size of quadratic term |W| = 3.08  -> boundary is curved


## Example 2: The Perceptron Learning Algorithm

The perceptron algorithm, line by line. We start from a deliberately bad line so that the first epoch has plenty of mistakes (11). The weights are nudged after each one, and by epoch 2 there are no mistakes, so training stops. Then we classify three new points by the sign of g(x). Look at the last line: the closest training point is only 0.032 units from the boundary. The perceptron stopped at the first line that worked, not the safest one. Keep that number in mind for Chapter 12.

In [2]:
# Chapter 11 - Example 2: The perceptron learning algorithm from scratch
import numpy as np

rng = np.random.default_rng(1)
X = np.vstack([rng.normal([1, 1], 0.5, (20, 2)), rng.normal([-1, -1], 0.5, (20, 2))])
y = np.r_[np.ones(20), -np.ones(20)]                 # labels +1 and -1
Xh = np.column_stack([np.ones(len(X)), X])           # x_hat = [1, x1, x2] so w0 is the bias

w = np.array([2.0, -1.0, 0.5])                       # 1. start from a deliberately bad line
eta = 0.1
for epoch in range(1, 101):
    mistakes = 0
    for xn, yn in zip(Xh, y):                        # 2. present each example
        s = w @ xn
        if yn * s <= 0:                              # 3. wrong side (or on the line) -> nudge w
            w += eta * yn * xn; mistakes += 1         #    +eta*x for class +1, -eta*x for -1
    print(f"Epoch {epoch}: misclassifications = {mistakes}")
    if mistakes == 0:                                # 5. stop when nothing is wrong
        break

print(f"\nLearned w = [w0, w1, w2] = {np.round(w, 3)}")
print(f"Boundary: {w[1]:.2f}*x1 + {w[2]:.2f}*x2 + {w[0]:.2f} = 0")
for t in ([1.5, 1.0], [-0.5, -1.5], [0.5, 0.4]):
    s = w @ np.r_[1, t]
    print(f"Test {t}: g(x) = {s:+.2f} -> class {'+1' if s > 0 else '-1'}")

# Distance of the closest training point to the line (the margin this perceptron got)
dist = np.abs(Xh @ w) / np.linalg.norm(w[1:])
print(f"Closest training point is {dist.min():.3f} units from the boundary")

Epoch 1: misclassifications = 11
Epoch 2: misclassifications = 0

Learned w = [w0, w1, w2] = [0.9   0.066 1.635]
Boundary: 0.07*x1 + 1.64*x2 + 0.90 = 0
Test [1.5, 1.0]: g(x) = +2.63 -> class +1
Test [-0.5, -1.5]: g(x) = -1.59 -> class -1
Test [0.5, 0.4]: g(x) = +1.59 -> class +1
Closest training point is 0.032 units from the boundary


## Example 3: Where the Perceptron Fails

The XOR pattern puts one class on two opposite corners and the other class on the other two. The perceptron keeps making mistakes epoch after epoch and never converges. Scikit-learn's perceptron scores only 42%, worse than guessing. K-NN, which learns a nonlinear boundary, scores 100%. With overlapping classes the perceptron also never reaches zero errors. These are exactly the gaps that SVMs (and later neural networks) fill.

In [3]:
# Chapter 11 - Example 3: Where the perceptron fails - data that a line cannot separate
import numpy as np
from sklearn.linear_model import Perceptron
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(0)
# XOR pattern: opposite corners belong to the same class
centers = [([0, 0], -1), ([1, 1], -1), ([0, 1], 1), ([1, 0], 1)]
X = np.vstack([rng.normal(c, 0.1, (25, 2)) for c, _ in centers])
y = np.repeat([lab for _, lab in centers], 25)

def train_perceptron(X, y, epochs=50, eta=0.1):
    Xh = np.column_stack([np.ones(len(X)), X]); w = np.zeros(3); history = []
    for _ in range(epochs):
        errs = 0
        for xn, yn in zip(Xh, y):
            if yn * (w @ xn) <= 0:
                w += eta * yn * xn; errs += 1
        history.append(errs)
        if errs == 0: break
    return w, history

w, hist = train_perceptron(X, y)
print("XOR data - mistakes per epoch (first 10):", hist[:10])
print(f"After {len(hist)} epochs still {hist[-1]} mistakes -> it never converges.")
print(f"Perceptron (scikit-learn) accuracy on XOR: "
      f"{Perceptron(random_state=0).fit(X, y).score(X, y):.2f}")

# A non-linear method handles it easily (K-NN learns a curved boundary implicitly)
print(f"K-NN (K=5) accuracy on XOR: {KNeighborsClassifier(5).fit(X, y).score(X, y):.2f}")

# Overlapping classes: even the best line makes some mistakes
Xo = np.vstack([rng.normal([0, 0], 1, (100, 2)), rng.normal([1.5, 1.5], 1, (100, 2))])
yo = np.r_[-np.ones(100), np.ones(100)]
w, hist = train_perceptron(Xo, yo)
print(f"\nOverlapping data: mistakes in the last 5 epochs {hist[-5:]} - never reaches 0.")

XOR data - mistakes per epoch (first 10): [3, 5, 4, 4, 4, 5, 5, 4, 4, 7]
After 50 epochs still 8 mistakes -> it never converges.
Perceptron (scikit-learn) accuracy on XOR: 0.42
K-NN (K=5) accuracy on XOR: 1.00

Overlapping data: mistakes in the last 5 epochs [9, 15, 9, 10, 13] - never reaches 0.
